# Notebook 13 — RAG-Enhanced Recommendations

**What you'll learn:**
- How the PINN knowledge base provides literature context
- BM25 search over 20 curated PDE knowledge entries
- How recommendations change with RAG augmentation
- Building custom search queries from PDESpec features

**Prerequisites:** Notebooks 10-12, RAG library

**Time:** ~30 minutes

## 1. What RAG Adds to PINN Recommendations

The rule-based PINN Agent (Notebook 10) applies hand-coded heuristics: "sharp gradients → more collocation", "high frequency → sinusoidal Ansatz". These rules are correct but **anonymous** — they don't say *why*, or cite the literature behind them.

Retrieval-Augmented Generation (RAG) connects recommendations to the published research:

```
PDESpec features
     ↓ build_search_query()
BM25 search over 20 knowledge entries
     ↓ top-k documents retrieved
Formatted context appended to ArchitectureRec
     ↓
knowledge_context field: specific techniques + failure modes
```

No vector database, no embeddings, no LLM call at retrieval time. Pure keyword matching over a structured store.

## 2. Loading the Knowledge Base

In [ ]:
from pathlib import Path
from rag import KnowledgeStore, SearchEngine

STORE_PATH = Path("../data/pinn-knowledge/store")

store = KnowledgeStore.load(STORE_PATH)
docs = store.list_documents()

print(f"Loaded {len(docs)} documents from {STORE_PATH}")
print()
print(f"{'doc_id':<40} {'pde_type':<20} techniques")
print("-" * 90)
for meta in docs:
    techs = ", ".join(meta.techniques[:2]) if meta.techniques else "—"
    print(f"{meta.doc_id:<40} {(meta.pde_type or '—'):<20} {techs}")

In [ ]:
# Build the BM25 search engine from the store
engine = SearchEngine.from_store(store)

print("SearchEngine ready.")
print(f"Corpus size: {len(engine._corpus)} documents indexed")

## 3. Searching the Knowledge Base

`SearchEngine.search()` uses BM25 (Best Match 25) — a probabilistic ranking function that scores each document by term frequency and inverse document frequency. No neural network needed.

In [ ]:
# Query 1: Burgers equation with shock formation
results = engine.search("burgers shock viscosity", top_k=5)
print("Query: 'burgers shock viscosity'")
print(f"Top {len(results)} results:")
for rank, doc_id in enumerate(results, 1):
    meta = store.get_metadata(doc_id)
    print(f"  {rank}. {doc_id}")
    print(f"     PDE type : {meta.pde_type or '(general)'}")
    print(f"     Techniques: {', '.join(meta.techniques) if meta.techniques else 'none'}")

In [ ]:
# Query 2: Spectral bias (F-principle) — a core PINN failure mode
results = engine.search("spectral bias high frequency", top_k=5)
print("Query: 'spectral bias high frequency'")
print(f"Top {len(results)} results:")
for rank, doc_id in enumerate(results, 1):
    meta = store.get_metadata(doc_id)
    print(f"  {rank}. {doc_id}")
    if meta.known_issues:
        print(f"     Known issues: {', '.join(meta.known_issues[:2])}")

In [ ]:
# Query 3: Loss weighting — common tuning challenge
results = engine.search("loss weighting", top_k=5)
print("Query: 'loss weighting'")
print(f"Top {len(results)} results:")
for rank, doc_id in enumerate(results, 1):
    print(f"  {rank}. {doc_id}")

## 4. Browsing Knowledge Entries

Each entry in the store is a `DocumentTree` with hierarchical nodes (sections, subsections). Metadata captures what the document is about at a glance.

In [ ]:
# Inspect the Burgers equation entry in detail
burgers_meta = store.get_metadata("burgers-equation")

print("=== Burgers Equation Entry ===")
print(f"Name        : {burgers_meta.doc_name}")
print(f"PDE type    : {burgers_meta.pde_type}")
print(f"Techniques  : {', '.join(burgers_meta.techniques)}")
print(f"Known issues: {', '.join(burgers_meta.known_issues)}")
print(f"Keywords    : {', '.join(burgers_meta.keywords[:8])}")
print(f"Nodes       : {burgers_meta.node_count}")
print(f"Tokens      : {burgers_meta.total_tokens}")

In [ ]:
# Browse the spectral bias entry — relevant for any high-frequency PDE
sb_meta = store.get_metadata("spectral-bias")

print("=== Spectral Bias Entry ===")
print(f"Name        : {sb_meta.doc_name}")
print(f"Techniques  : {', '.join(sb_meta.techniques)}")
print(f"Known issues: {', '.join(sb_meta.known_issues)}")
print(f"Keywords    : {', '.join(sb_meta.keywords[:8])}")
print()

# Look at the tree structure
sb_tree = store.get_document("spectral-bias")
print("Document sections:")
for node in sb_tree.root_nodes:
    print(f"  [{node.level}] {node.title}")
    for child in node.children:
        print(f"      [{child.level}] {child.title}")

## 5. RAG-Enhanced PINN Agent

Now we see the payoff. The `PINNAgent` can accept a `knowledge_store_dir` argument. When it does, every call to `recommend()` runs BM25 retrieval and attaches the results to `ArchitectureRec.knowledge_context`.

We'll compare **without** and **with** knowledge store on the same Burgers equation spec.

In [ ]:
from lang_pinn import PDESpec, PINNAgent

# Burgers equation: u_t + u*u_x = nu*u_xx
# Archetypal nonlinear PDE — forms shocks when viscosity (nu) is small
burgers = PDESpec(
    name="Burgers Equation",
    equation="u_t + u*u_x = nu*u_xx",
    independent_vars=["x", "t"],
    dependent_var="u",
    order=2,
    spatial_dim=1,
    domain={"x": (-1.0, 1.0), "t": (0.0, 1.0)},
    initial_conditions=["u(x, 0) = -sin(pi*x)"],
    boundary_conditions=["u(-1, t) = 0", "u(1, t) = 0"],
    parameters={"nu": 0.01 / 3.14159},
    is_linear=False,
    is_time_dependent=True,
    has_sharp_gradients=True,  # low viscosity -> shock forms near t=1
)

print(f"PDE: {burgers.name}")
print(f"Equation: {burgers.equation}")
print(f"Sharp gradients: {burgers.has_sharp_gradients}")
print(f"Nonlinear: {not burgers.is_linear}")

In [ ]:
# --- Without knowledge store ---
agent_basic = PINNAgent()  # no knowledge_store_dir
rec_basic = agent_basic.recommend(burgers)

print("=== Basic Recommendation (no RAG) ===")
print(f"Architecture : {rec_basic.hidden_layers}x{rec_basic.hidden_neurons} {rec_basic.activation}")
print(f"Epochs       : {rec_basic.epochs}")
print(f"Collocation  : {rec_basic.n_collocation}")
print(f"Loss weights : {rec_basic.loss_weights}")
print(f"Ansatz       : {rec_basic.use_ansatz}")
print()
print(f"Reasoning: {rec_basic.reasoning}")
print()
print(f"Knowledge context: {repr(rec_basic.knowledge_context[:80]) if rec_basic.knowledge_context else '(empty)'}")

In [ ]:
# --- With knowledge store ---
agent_rag = PINNAgent(knowledge_store_dir=STORE_PATH)
rec_rag = agent_rag.recommend(burgers)

print("=== RAG-Enhanced Recommendation ===")
print(f"Architecture : {rec_rag.hidden_layers}x{rec_rag.hidden_neurons} {rec_rag.activation}")
print(f"Epochs       : {rec_rag.epochs}")
print(f"Collocation  : {rec_rag.n_collocation}")
print(f"Loss weights : {rec_rag.loss_weights}")
print(f"Ansatz       : {rec_rag.use_ansatz}")
print()
print(f"Reasoning: {rec_rag.reasoning}")
print()
print(f"Knowledge context present: {bool(rec_rag.knowledge_context)}")
print(f"Context length: {len(rec_rag.knowledge_context)} chars")

The hyperparameters are identical — RAG does not change the rule-based logic. What changes is `knowledge_context`: a formatted block of literature text attached to the recommendation so you (or an LLM in hybrid mode) can trace the *why* behind each choice.

## 6. Examining the Knowledge Context

Let's look at what was actually retrieved.

In [ ]:
# Print the full retrieved context
print(rec_rag.knowledge_context)

In [ ]:
# Parse it to highlight specific signals the recommendation is grounded in
context = rec_rag.knowledge_context

signals = [
    "RAR",
    "causal training",
    "adaptive",
    "shock",
    "collocation",
    "viscosity",
    "failure",
    "spectral bias",
    "loss weight",
]

print("Signals present in retrieved context:")
for signal in signals:
    found = signal.lower() in context.lower()
    marker = "[YES]" if found else "[ no]"
    print(f"  {marker}  {signal}")

## 7. Custom Search Queries from PDESpec

`build_search_query()` translates a `PDESpec`'s feature flags into a search string. This is what the agent runs internally, but you can call it directly to inspect or override.

In [ ]:
from lang_pinn.agents.knowledge import build_search_query, load_knowledge, search_knowledge

query = build_search_query(burgers)
print("Auto-generated query for Burgers:")
print(f"  '{query}'")
print()
print("How feature flags map to query terms:")
print(f"  spec.name                    -> '{burgers.name}'")
print(f"  spec.equation                -> '{burgers.equation}'")
print(f"  has_sharp_gradients=True     -> 'sharp gradients shock adaptive refinement'")
print(f"  is_linear=False              -> 'nonlinear'")
print(f"  is_time_dependent=True       -> 'time dependent'")

In [ ]:
# You can also call search_knowledge directly with a custom query
store2, engine2 = load_knowledge(STORE_PATH)

# Try a more targeted query
custom_query = "causal training time-dependent nonlinear shock"
context_custom = search_knowledge(custom_query, store2, engine2, top_k=2)

print(f"Custom query: '{custom_query}'")
print(f"Retrieved context ({len(context_custom)} chars):")
print()
print(context_custom[:1200] + ("..." if len(context_custom) > 1200 else ""))

In [ ]:
# Compare queries: what changes when has_high_frequency is set?
from lang_pinn import PDESpec

helmholtz = PDESpec(
    name="Helmholtz Equation",
    equation="u_xx + u_yy + k^2*u = f(x,y)",
    independent_vars=["x", "y"],
    dependent_var="u",
    order=2,
    spatial_dim=2,
    domain={"x": (0.0, 1.0), "y": (0.0, 1.0)},
    boundary_conditions=["u = 0 on boundary"],
    is_linear=True,
    is_time_dependent=False,
    has_high_frequency=True,  # large k -> oscillatory solution
)

q_helmholtz = build_search_query(helmholtz)
print("Helmholtz query:")
print(f"  '{q_helmholtz}'")
print()

results_h = engine2.search(q_helmholtz, top_k=4)
print("Top results:")
for i, doc_id in enumerate(results_h, 1):
    meta = store2.get_metadata(doc_id)
    print(f"  {i}. {doc_id}  ({meta.pde_type or 'general'})")

### What the feature flags control

| Flag | Query terms added | Retrieved entries |
|------|-------------------|-------------------|
| `has_sharp_gradients=True` | `sharp gradients shock adaptive refinement` | burgers-equation, collocation-strategies |
| `has_high_frequency=True` | `high frequency oscillation spectral bias` | spectral-bias, helmholtz-equation, high-frequency-odes |
| `has_periodic_bc=True` | `periodic boundary conditions` | periodic-boundary-conditions, advection-equation |
| `is_linear=False` | `nonlinear` | burgers-equation, navier-stokes-2d, allen-cahn-equation |
| `is_time_dependent=True` | `time dependent` | burgers-equation, heat-equation, advection-equation |
| `spatial_dim >= 2` | `2D multi-dimensional` | navier-stokes-2d, poisson-equation |

## 8. Using Knowledge Context in the Full Workflow

In hybrid mode (Notebook 11), the LLM sees the `knowledge_context` alongside the architecture recommendation. Here's what that looks like end-to-end:

In [ ]:
# Simulate what the hybrid mode orchestrator passes to the LLM reviewer
def format_recommendation_for_review(spec: PDESpec, rec) -> str:
    """Show the full context a reviewer (human or LLM) receives."""
    lines = [
        f"PDE: {spec.name}",
        f"Equation: {spec.equation}",
        "",
        "Architecture:",
        f"  {rec.hidden_layers}x{rec.hidden_neurons} {rec.activation}",
        f"  {rec.n_collocation} collocation points",
        f"  {rec.epochs} epochs, lr={rec.learning_rate}",
        f"  Loss weights: {rec.loss_weights}",
        "",
        f"Reasoning: {rec.reasoning}",
    ]
    if rec.knowledge_context:
        lines += [
            "",
            "--- Literature Context (RAG) ---",
            rec.knowledge_context[:600] + "..." if len(rec.knowledge_context) > 600 else rec.knowledge_context,
        ]
    return "\n".join(lines)

print(format_recommendation_for_review(burgers, rec_rag))

## Exercise: Try with a Different PDE

Build a `PDESpec` for one of these and inspect what the knowledge base retrieves:

1. **Schrödinger equation** — complex-valued, oscillatory (`has_high_frequency=True`, `output_dim=2`)
2. **2D Navier-Stokes** — nonlinear, 2D spatial (`spatial_dim=2`, `is_linear=False`)
3. **Allen-Cahn** — sharp interface problem (`has_sharp_gradients=True`, nonlinear)
4. **Your own PDE** — set the feature flags that apply

For each: check what documents are retrieved and whether `knowledge_context` mentions specific techniques you recognise from the literature.

In [ ]:
# Your turn!
# my_spec = PDESpec(
#     name="...",
#     equation="...",
#     independent_vars=[...],
#     dependent_var="u",
#     order=...,
#     spatial_dim=...,
#     domain={...},
#     has_sharp_gradients=...,
#     has_high_frequency=...,
# )
#
# query = build_search_query(my_spec)
# print(f"Query: {query}")
#
# results = engine.search(query, top_k=5)
# print(f"Retrieved: {results}")
#
# agent = PINNAgent(knowledge_store_dir=STORE_PATH)
# rec = agent.recommend(my_spec)
# print(rec.knowledge_context)

## Summary: The Complete PINN Curriculum

You've now seen how retrieval-augmented generation connects the rule-based agent to curated PDE literature.

| Notebook | Topic | Key Skill |
|----------|-------|-----------|
| 01 | What are PINNs? | Motivation, landscape |
| 02 | Automatic differentiation | `torch.autograd.grad` |
| 03 | First PINN from scratch | Raw PyTorch PINN |
| 04 | Data vs Physics vs Hybrid | Loss function design |
| 05 | PDEs and boundary conditions | Multi-term losses |
| 06 | Training tricks | Ansatz, weighting, scheduling |
| 07 | Parametric and inverse | Parameters as inputs |
| 08 | Honest assessment | When (not) to use PINNs |
| 09 | Inverse Navier-Stokes | Advanced: Re inference |
| 10 | Lang-PINN intro | 3 agents, 3 modes |
| 11 | Hybrid mode | Feedback loop, quality scoring |
| 12 | Bring your own PDE | End-to-end workflow |
| **13** | **RAG-enhanced recommendations** | **BM25 search, knowledge_context** |
| **14** | **RAG pipeline deep dive** | **File ingestion, deduplication, indexing** |